## Import

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.6 MB/s eta 0:00:00


In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 43.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [ ]:
#fisheye
from roboflow import Roboflow
rf = Roboflow(api_key="PqVVZQ5lGpLIWL0ApaJQ")
project = rf.workspace("stubbornstrawsberries").project("fisheye8k")
version = project.version(7)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Fisheye8K-7 in yolov11:: 100%|██████████| 8654/8654 [00:02<00:00, 3346.74it/s]


In [ ]:
## fish eye human
'''
from roboflow import Roboflow
rf = Roboflow(api_key="PqVVZQ5lGpLIWL0ApaJQ")
project = rf.workspace("meo-4zf6i").project("fisheye-tphj8-5ecuj")
version = project.version(1)
dataset = version.download("yolov11")
'''

'\nfrom roboflow import Roboflow\nrf = Roboflow(api_key="PqVVZQ5lGpLIWL0ApaJQ")\nproject = rf.workspace("meo-4zf6i").project("fisheye-tphj8-5ecuj")\nversion = project.version(1)\ndataset = version.download("yolov11")\n'

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Stripping image

In [ ]:
# === CONFIGURATION ===
images_dir = "/content/Fisheye8K-7/train/images"
labels_dir = "/content/Fisheye8K-7/train/labels"

# Output paths for each class
paths = {
    0: "/content/drive/MyDrive/ML/dtset/fisheye/0",
    1: "/content/drive/MyDrive/ML/dtset/fisheye/1",
    2: "/content/drive/MyDrive/ML/dtset/fisheye/2",
    3: "/content/drive/MyDrive/ML/dtset/fisheye/3",
    4: "/content/drive/MyDrive/ML/dtset/fisheye/4",
}

# Make sure the class folders exist
for p in paths.values():
    os.makedirs(p, exist_ok=True)

# === PROCESS IMAGES ===
for filename in os.listdir(images_dir):
    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(images_dir, filename)
    label_path = os.path.join(labels_dir, os.path.splitext(filename)[0] + ".txt")

    if not os.path.exists(label_path):
        continue

    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print(f"⚠️ Skipping unreadable image: {filename}")
        continue

    h, w, _ = img.shape

    # Read label file
    with open(label_path, "r") as f:
        lines = f.readlines()

    # Process each bounding box
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) != 5:
            continue  # skip invalid lines

        cls, x_center, y_center, box_w, box_h = map(float, parts)
        cls = int(cls)

        # Convert from normalized (0–1) to pixel coordinates
        x1 = int((x_center - box_w / 2) * w)
        y1 = int((y_center - box_h / 2) * h)
        x2 = int((x_center + box_w / 2) * w)
        y2 = int((y_center + box_h / 2) * h)

        # Clip coordinates to image boundaries
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        # Crop bounding box
        cropped = img[y1:y2, x1:x2]
        if cropped.size == 0:
            continue

        # Determine output path based on class
        if cls in paths:
            save_dir = paths[cls]
            out_filename = f"{os.path.splitext(filename)[0]}_obj{i}_cls{cls}.jpg"
            out_path = os.path.join(save_dir, out_filename)
            cv2.imwrite(out_path, cropped)
        else:
            print(f"⚠️ Unknown class {cls} in {filename}")

print("✅ All bounding boxes cropped and saved into class folders!")

## Train original dtset

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil, glob

DATA_YAML = "/content/Fisheye8K-7/data.yaml"

model = YOLO("yolo11m.pt")

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=-1,  # <- AutoBatch (or replace with int like 8 or 16)
    device=0   # use 'cpu' if no GPU
)

# Save/export the trained weights
# Option A: copy the best checkpoint that YOLO already saved
best = sorted(glob.glob("runs/detect/train*/weights/best.pt"))[-1]
shutil.copy(best, "yolov11m_fisheye8k.pt")

# Option B: also persist the in-memory model object (optional)
model.save("yolov11m_fisheye8k_latest.pt")


Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Fisheye8K-7/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12

## CR2

In [ ]:
from google.colab import files

files.download("yolov11m_fisheye8k.pt")
files.download("yolov11m_fisheye8k_latest.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.metrics import classification_report
from ultralytics import YOLO
import glob
import os
import numpy as np

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
bestpt = "/content/drive/MyDrive/ML/dtset/fisheye/DL/FINAL.pt"
test_images_path = "/content/Fisheye8K-7/test/images"
test_labels_path = "/content/Fisheye8K-7/test/labels"
model = YOLO(bestpt)

In [ ]:
results = model.val(
    data="/content/Fisheye8K-7/data.yaml",  # path to your yaml
    split="test",                            # use test set
    save_json=True,                          # save COCO-style results
    conf=0.25                                # confidence threshold (adjust as needed)
)
##/content/drive/MyDrive/ML/dtset/fisheye/DL/yolov11m_fisheye8k_latest.pt

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1236.0±282.1 MB/s, size: 53.4 KB)
val: Scanning /content/Fisheye8K-7/test/labels... 255 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 255/255 1.8Kit/s 0.1s
val: New cache created: /content/Fisheye8K-7/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 1.5it/s 11.0s
                   all        255       4018      0.933      0.874      0.932      0.718
                     0         58         77      0.963      0.857      0.929      0.732
                     1        252       2383      0.945      0.911      0.953       0.69
                     2        246       1245      0.942      0.931      0.965      0.753
                     3         85        218      0.888      0.765      0.869      0.535
                     4         89         95      0.928      0.905   

In [ ]:
##/content/drive/MyDrive/ML/dtset/fisheye/DL/yolov11m_fisheye8k_latest.pt
print(results.box.map)      # mAP@0.5:0.95
print(results.box.map50)    # mAP@0.5
print(results.box.map75)    # mAP@0.75
print(results.box.maps)     # per-class mAPs


0.7180092890874374
0.9320318429337313
0.8119711312778053
[    0.73168     0.69024     0.75344     0.53482     0.87986]


In [ ]:
## /content/drive/MyDrive/ML/dtset/fisheye/DL/FINAL.pt
print(results.box.map)      # mAP@0.5:0.95
print(results.box.map50)    # mAP@0.5
print(results.box.map75)    # mAP@0.75
print(results.box.maps)

In [ ]:
import yaml

with open("/content/Fisheye8K-7/data.yaml", "r") as f:
    data = yaml.safe_load(f)

class_names = data["names"]
print(class_names)


['0', '1', '2', '3', '4']


In [ ]:
# Assuming `results` is from model.val()
for i, ap in enumerate(results.box.maps):
    print(f"Class {i} ({class_names[i]}): AP = {ap:.4f}")


Class 0 (0): AP = 0.7317
Class 1 (1): AP = 0.6902
Class 2 (2): AP = 0.7534
Class 3 (3): AP = 0.5348
Class 4 (4): AP = 0.8799
